# RINEX 3 GPS observations: what does the **third character** mean?

**Goal (10–15 minutes):** read the observation names in our code-positioning notebooks:

`CODE1 = ('C1W', 'C1C', 'C1X', 'C1L')`

`CODE2 = ('C2W', 'C2C', 'C2D', 'C2P', 'C2X', 'C2S', 'C2L')`

The last character is **not a quality grade**. It specifies a signal component or a receiver tracking method. This notebook covers **GPS only**; the same letter can mean something different for Galileo or another constellation.

## 1. The three positions: `t n a`

| Position | Meaning | GPS examples |
|:--|:--|:--|
| **1st: `t`** | Observation type: `C` pseudorange, `L` carrier phase, `D` Doppler, `S` signal strength | `C1W` vs `L1W` |
| **2nd: `n`** | Frequency band | `1` = L1, `2` = L2, `5` = L5 |
| **3rd: `a`** | **Tracking attribute**: which signal/code/component or method | `C`, `W`, `L`, `X`, ... |

GPS frequency bands: **L1 = 1575.42 MHz**, **L2 = 1227.60 MHz**, **L5 = 1176.45 MHz**.

Compare `C1W` and `L1W`: both use the **same band and tracking attribute**, but the first is a *code range* (metres) and the second is a *carrier-phase observation* (cycles).

**Notation:** `C[125]?` is a convenient *shell glob* (where `?` means one character). It is **not** the corresponding regular expression; in a regex use `C[125][A-Z]`. Only valid band/attribute combinations from the RINEX tables are permitted.

## 2. GPS L1: what does `C1?` or `L1?` mean?

| Final letter | GPS signal or tracking mode on L1 | Example |
|:--|:--|:--|
| **`C`** | Traditional civilian **C/A** signal | `C1C` |
| **`W`** | **Z-tracking** of encrypted P(Y) under anti-spoofing | `C1W` |
| **`P`** | P-code tracking (anti-spoofing off) | `C1P` |
| **`S`** | Modern **L1C data** component (D) | `C1S` |
| **`L`** | Modern **L1C pilot** component (P) | `C1L` |
| **`X`** | Combined **L1C data + pilot** (D+P) | `C1X` |

For example, `C1C` and `C1X` are both L1 pseudoranges, **but they were obtained from different signals**.

Also notice: in **`L1L`**, the **first `L`** means *carrier phase*, while the **last `L`** means *L1C pilot*. The letters occupy different positions and have different jobs.

## 3. GPS L2: why the list of alternatives is longer

| Final letter | GPS signal or tracking method on L2 | Example |
|:--|:--|:--|
| **`W`** | Z-tracking of encrypted P(Y) | `C2W` |
| **`P`** | P-code (anti-spoofing off) | `C2P` |
| **`C`** | C/A | `C2C` |
| **`D`** | **Semi-codeless** L1(C/A) + (P2 − P1) method | `C2D` |
| **`S`** | Civil L2C **M** component | `C2S` |
| **`L`** | Civil L2C **L** component | `C2L` |
| **`X`** | Combined L2C **M + L** | `C2X` |

**Important distinction:** GPS `C2C` is the L2 **C/A** entry in the RINEX table; **`C2S`**, not `C2C`, denotes the L2C **M** component. Do not infer a signal just from the letter `C`.

For completeness, GPS L5 uses `I` (in-phase/data), `Q` (quadrature/pilot), and `X` (combined I+Q): for example `C5I`, `C5Q`, `C5X`.

In [1]:
# A tiny GPS-only dictionary. No RINEX file or other package is required.
GPS_SIGNALS = {
    '1': {'C': 'C/A', 'W': 'P(Y), Z-tracking', 'P': 'P-code',
          'S': 'L1C data', 'L': 'L1C pilot', 'X': 'L1C data+pilot'},
    '2': {'W': 'P(Y), Z-tracking', 'P': 'P-code', 'C': 'C/A',
          'D': 'semi-codeless', 'S': 'L2C M', 'L': 'L2C L',
          'X': 'L2C M+L'},
    '5': {'I': 'L5 I', 'Q': 'L5 Q', 'X': 'L5 I+Q'},
}
BANDS = {'1': 'GPS L1', '2': 'GPS L2', '5': 'GPS L5'}
OBS_TYPES = {'C': 'pseudorange', 'L': 'carrier phase',
             'D': 'Doppler', 'S': 'signal strength'}

def explain_gps_observation(name):
    if len(name) != 3:
        raise ValueError('Expected exactly three RINEX observation characters')
    kind, band, attribute = name.upper()
    if kind not in OBS_TYPES or attribute not in GPS_SIGNALS.get(band, {}):
        raise ValueError(f'{name!r} is not included in this small teaching table')
    return f'{name}: {OBS_TYPES[kind]}, {BANDS[band]}, {GPS_SIGNALS[band][attribute]}'

for name in ('C1C', 'C1W', 'C1L', 'C1X', 'L1X', 'C2D', 'C2S', 'C2X', 'C5I', 'L5Q'):
    print(explain_gps_observation(name))


C1C: pseudorange, GPS L1, C/A
C1W: pseudorange, GPS L1, P(Y), Z-tracking
C1L: pseudorange, GPS L1, L1C pilot
C1X: pseudorange, GPS L1, L1C data+pilot
L1X: carrier phase, GPS L1, L1C data+pilot
C2D: pseudorange, GPS L2, semi-codeless
C2S: pseudorange, GPS L2, L2C M
C2X: pseudorange, GPS L2, L2C M+L
C5I: pseudorange, GPS L5, L5 I
L5Q: carrier phase, GPS L5, L5 Q


## 4. Why the positioning notebook tries several alternatives

Our original `CODE1` and `CODE2` tuples are **ordered fallback lists**. They mean: “Try a valid pseudorange from this frequency band.” They do **not** mean that the measurements are numerically interchangeable.

- `C1W`, `C1C` and `C1X` are all on GPS L1, but their signals/tracking differ.
- `C2W` and `C2X` are both on GPS L2, but their signals/tracking differ.
- **Code biases (OSB/DCB)** can differ between signal types. Record **which actual observation code** was selected so its corresponding bias can be applied.
- A RINEX header lists what types the receiver records. An individual satellite/epoch may still have a missing measurement.
- If we later use carrier phase, the same attribute rules apply to `L1?`, `L2?`, `L5?`; phase ambiguities and phase biases require separate treatment.

This is why the positioning notebooks retain the `ctype1`/`ctype2` returned by `fetch(...)` before requesting corrections.

In [2]:
CODE1 = ('C1W', 'C1C', 'C1X', 'C1L')
CODE2 = ('C2W', 'C2C', 'C2D', 'C2P', 'C2X', 'C2S', 'C2L')

print('Preferred GPS L1 pseudorange choices:')
for name in CODE1:
    print(' ', explain_gps_observation(name))

print('\nPreferred GPS L2 pseudorange choices:')
for name in CODE2:
    print(' ', explain_gps_observation(name))

# Each list selects *one frequency band*, not one uniform signal type.
assert {code[1] for code in CODE1} == {'1'}
assert {code[1] for code in CODE2} == {'2'}


Preferred GPS L1 pseudorange choices:
  C1W: pseudorange, GPS L1, P(Y), Z-tracking
  C1C: pseudorange, GPS L1, C/A
  C1X: pseudorange, GPS L1, L1C data+pilot
  C1L: pseudorange, GPS L1, L1C pilot

Preferred GPS L2 pseudorange choices:
  C2W: pseudorange, GPS L2, P(Y), Z-tracking
  C2C: pseudorange, GPS L2, C/A
  C2D: pseudorange, GPS L2, semi-codeless
  C2P: pseudorange, GPS L2, P-code
  C2X: pseudorange, GPS L2, L2C M+L
  C2S: pseudorange, GPS L2, L2C M
  C2L: pseudorange, GPS L2, L2C L


## 5. Optional: inspect the real DYNG RINEX header

This section only looks at the observation types advertised in the header. It does not process the whole RINEX file or perform positioning.

The example file is expected in the course's shared `data/` directory. This notebook lives in `JupyterLab/gnss/`, so the usual relative path is `../../data`.

In [3]:
from os.path import join, isfile

data_path = "../../data"
rinex_path = join(data_path, "DYNG00GRC_R_20240010000_01D_30S_MO.rnx")

if not isfile(rinex_path):
    print(f'Optional example unavailable: {rinex_path}')
    print('Place the course RINEX file in data/ to run this cell.')
else:
    from dsoclasses.rinex.gnss.rinex import GnssRinex

    rnx = GnssRinex(rinex_path)
    available = rnx.obscodes.get('G', [])
    print('GPS pseudorange types in the RINEX header:')
    print([name for name in available if name.startswith('C')])

    # The order is an intentional preference, not a claim of equivalence.
    for label, candidates in (('L1', CODE1), ('L2', CODE2)):
        found = next((name for name in candidates if name in available), None)
        print(f'{label}: first advertised candidate = {found}')
        if found:
            print('  ', explain_gps_observation(found))
    print('Note: a listed type can still be missing for a particular satellite/epoch.')


GPS pseudorange types in the RINEX header:
['C1C', 'C1W', 'C2W', 'C2L', 'C5Q', 'C1L']
L1: first advertised candidate = C1W
   C1W: pseudorange, GPS L1, P(Y), Z-tracking
L2: first advertised candidate = C2W
   C2W: pseudorange, GPS L2, P(Y), Z-tracking
Note: a listed type can still be missing for a particular satellite/epoch.


## Take-away and two quick questions

1. What is the difference between **`C1X`** and **`L1X`**? *(Observation type only: code range versus carrier phase.)*
2. What is the difference between **`C2S`** and **`C2X`**? *(L2C M component versus combined M+L tracking.)*

**Rule to remember:** the *last character must be interpreted together with the constellation and frequency band*. Do not automatically equate `X`, `L`, `C`, or `W` across different bands or satellite systems.

**Source:** IGS/RTCM RINEX Working Group, *RINEX 3.05*, §5.2.17, Tables 13–14:
[official specification](https://files.igs.org/pub/data/format/rinex305.pdf).